# KAN Hyperparameter Optimization

Optuna searches the five selected KAN hyperparameters and saves every completed trial in one timestamped results directory.

In [ ]:
import os
import sys
from datetime import datetime

from importlib import reload
current_dir = os.getcwd()
utilities_dir = os.path.join(current_dir, '../../utils')
os.chdir(current_dir)
if utilities_dir not in sys.path:
    sys.path.insert(0, utilities_dir)
import plotting
import pinns
import infinite
reload(plotting)
reload(pinns)
reload(infinite)
import numpy as np
import sympy as sp
from calflops import calculate_flops
import pandas as pd
import joblib
import matplotlib.pyplot as plt 
import torch
import torch.nn as nn
import torch.optim as optim
from pinns import  MLP, init_weights, CoefficientNet, pde_loss_inf, observation_loss_u, observation_loss_k, train_dual_network, build_models, set_seed,run_experiment_inf,build_models_KAN
from pinns import build_models
from infinite import analytical_solution_inf, coefficient_inf, source_term_inf, generate_dataset_inf, evaluate_model_inf
torch.set_default_dtype(torch.float32)
from plotting import plot_histories_comparison

set_seed(1)
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

/home/orincon/miniconda3/envs/PIKAN-unbounded-domains-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Optuna Search Configuration

In [2]:
import optuna

# Search space from the architecture-matching study.
KAN_SEARCH_SPACE = {
    "hidden_layers": [1, 2, 3],
    "hidden_units": [15, 25, 35],
    "grid_size": [3, 5, 7],
    "spline_order": [2, 3, 4],
    "learning_rate": [1e-4, 1e-3, 1e-2],
}

N_TRIALS = 50
ADAM_ITERS = 2000
LBFGS_ITERS = 2000

# One directory contains the CSV summary and all saved trial models.
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
results_dir = f"results_kan_optuna_{timestamp}"
os.makedirs(results_dir, exist_ok=True)

print(f"Results will be saved to: {results_dir}")
print(f"Optuna trials: {N_TRIALS}")

Results will be saved to: results_kan_optuna_2026-09-14_21-57-58
Optuna trials: 50


## Objective Function

In [3]:
def objective(trial):
    """Run one KAN training configuration and return mean global error."""
    config = {
        "hidden_layers": trial.suggest_categorical(
            "hidden_layers",
            KAN_SEARCH_SPACE["hidden_layers"],
        ),
        "hidden_units": trial.suggest_categorical(
            "hidden_units",
            KAN_SEARCH_SPACE["hidden_units"],
        ),
        "grid_size": trial.suggest_categorical(
            "grid_size",
            KAN_SEARCH_SPACE["grid_size"],
        ),
        "spline_order": trial.suggest_categorical(
            "spline_order",
            KAN_SEARCH_SPACE["spline_order"],
        ),
        "learning_rate": trial.suggest_categorical(
            "learning_rate",
            KAN_SEARCH_SPACE["learning_rate"],
        ),
    }

    print(
        f"\n--- Trial {trial.number}: "
        f"L={config['hidden_layers']}, "
        f"N={config['hidden_units']}, "
        f"grid={config['grid_size']}, "
        f"order={config['spline_order']}, "
        f"lr={config['learning_rate']:.0e} ---"
    )

    try:
        err_u, err_k, compute_time = run_experiment_inf(
            model_type="KAN",
            hidden_layers=config["hidden_layers"],
            hidden_units=config["hidden_units"],
            grid_size=config["grid_size"],
            spline_order=config["spline_order"],
            adam_lr=config["learning_rate"],
            device=device,
            adam_iters=ADAM_ITERS,
            lbfgs_iters=LBFGS_ITERS,
            results_dir=results_dir,
        )
    except Exception as error:
        print(f"Trial {trial.number} failed: {error}")
        raise optuna.exceptions.TrialPruned()

    mean_global_error = 0.5 * (err_u + err_k)
    trial.set_user_attr("err_u", float(err_u))
    trial.set_user_attr("err_k", float(err_k))
    trial.set_user_attr("compute_time_sec", float(compute_time))

    print(
        f"Success! Time: {compute_time:.2f}s | "
        f"Err U: {err_u:.3e} | Err K: {err_k:.3e} | "
        f"Mean error: {mean_global_error:.3e}"
    )
    return mean_global_error

## Run Optimization

In [4]:
sampler = optuna.samplers.TPESampler(seed=1)
study = optuna.create_study(
    direction="minimize",
    sampler=sampler,
    study_name=f"kan_infinite_domain_{timestamp}",
)

study.optimize(
    objective,
    n_trials=N_TRIALS,
    catch=(RuntimeError, ValueError),
)

print("\n========================================")
print("BEST KAN CONFIGURATION")
print("========================================")
print(f"Mean global error: {study.best_value:.6e}")
print("Parameters:")
for name, value in study.best_params.items():
    print(f"  {name}: {value}")

[I 2026-09-14 21:58:03,271] A new study created in memory with name: kan_infinite_domain_2026-09-14_21-57-58



--- Trial 0: L=2, N=15, grid=7, order=4, lr=1e-03 ---


/home/orincon/miniconda3/envs/PIKAN-unbounded-domains-env/lib/python3.12/site-packages/torch/autograd/graph.py:869: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:335.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
[I 2026-09-14 22:04:12,844] Trial 0 finished with value: 0.004216520159221698 and parameters: {'hidden_layers': 2, 'hidden_units': 15, 'grid_size': 7, 'spline_order': 4, 'learning_rate': 0.001}. Best is trial 0 with value: 0.004216520159221698.



[KAN] L=2, N=15 | Params: 7,020 | Mean Err: 4.217e-03 | Saved to 'results_kan_optuna_2026-09-14_21-57-58/'.
Success! Time: 369.56s | Err U: 6.694e-03 | Err K: 1.739e-03 | Mean error: 4.217e-03

--- Trial 1: L=1, N=35, grid=3, order=3, lr=1e-02 ---


[I 2026-09-14 22:07:39,115] Trial 1 finished with value: 0.19600324322053717 and parameters: {'hidden_layers': 1, 'hidden_units': 35, 'grid_size': 3, 'spline_order': 3, 'learning_rate': 0.01}. Best is trial 0 with value: 0.004216520159221698.


...


[KAN] L=3, N=25 | Params: 26,500 | Mean Err: 3.516e-03 | Saved to 'results_kan_optuna_2026-09-14_21-57-58/'.
Success! Time: 376.71s | Err U: 2.883e-03 | Err K: 4.150e-03 | Mean error: 3.516e-03

--- Trial 48: L=3, N=25, grid=5, order=4, lr=1e-02 ---


[I 2026-09-15 02:43:48,472] Trial 48 finished with value: 0.001990392777380179 and parameters: {'hidden_layers': 3, 'hidden_units': 25, 'grid_size': 5, 'spline_order': 4, 'learning_rate': 0.01}. Best is trial 21 with value: 0.0010855430343352935.



[KAN] L=3, N=25 | Params: 29,150 | Mean Err: 1.990e-03 | Saved to 'results_kan_optuna_2026-09-14_21-57-58/'.
Success! Time: 476.62s | Err U: 1.930e-03 | Err K: 2.051e-03 | Mean error: 1.990e-03

--- Trial 49: L=3, N=25, grid=3, order=4, lr=1e-04 ---


[I 2026-09-15 02:51:44,164] Trial 49 finished with value: 0.0035985492341219978 and parameters: {'hidden_layers': 3, 'hidden_units': 25, 'grid_size': 3, 'spline_order': 4, 'learning_rate': 0.0001}. Best is trial 21 with value: 0.0010855430343352935.



[KAN] L=3, N=25 | Params: 23,850 | Mean Err: 3.599e-03 | Saved to 'results_kan_optuna_2026-09-14_21-57-58/'.
Success! Time: 475.68s | Err U: 5.158e-03 | Err K: 2.039e-03 | Mean error: 3.599e-03

BEST KAN CONFIGURATION
Mean global error: 1.085543e-03
Parameters:
  hidden_layers: 3
  hidden_units: 25
  grid_size: 5
  spline_order: 4
  learning_rate: 0.01


In [ ]:
 

# Persist the complete study and a tabular summary for later analysis.
data_dir = os.path.join(results_dir, "data")
os.makedirs(data_dir, exist_ok=True)

joblib.dump(study, os.path.join(data_dir, "study.pkl"))
joblib.dump(study, os.path.join(data_dir, f"study_{timestamp}.pkl"))

study_df = study.trials_dataframe()
study_csv_path = os.path.join(data_dir, "study.csv")
study_df.to_csv(study_csv_path, index=False)

# Keep the requested architecture while retaining every other trial column.
filtered_df = study_df[
    (study_df["params_hidden_layers"] == 3)
    & (study_df["params_hidden_units"] == 25)
].sort_values(by="value", ascending=True)
filtered_csv_path = os.path.join(data_dir, "study_filtered_sorted.csv")
filtered_df.to_csv(filtered_csv_path, index=False)

print(f"Saved study to: {data_dir}")
print(f"Saved trial summary to: {study_csv_path}")
print(f"Saved filtered summary to: {filtered_csv_path}")

Saved study to: results_kan_optuna_2026-09-14_20-51-03/data
Saved trial summary to: results_kan_optuna_2026-09-14_20-51-03/data/study.csv
Saved filtered summary to: results_kan_optuna_2026-09-14_20-51-03/data/study_filtered_sorted.csv
